<h3>Association Rule: Market Basket Analysis</h3>


In [2]:
data = "https://raw.githubusercontent.com/swapnilsaurav/AIML/refs/heads/main/ML/association/association_rule_market_basket.csv"

In [3]:
# mlxtend to import libraries
!pip install mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 8.5 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [7]:
df = pd.read_csv(data)
print(df.head(6))
print("Number of transactions = ",len(df))

  Transaction_ID                                  Items_Purchased  Item_Count
0          T0001                    Biscuits, Bread, Butter, Milk           4
1          T0002                      Butter, Pasta, Tomato Sauce           3
2          T0003                             Biscuits, Sugar, Tea           3
3          T0004        Bread, Butter, Milk, Pasta, Sugar, Yogurt           6
4          T0005               Biscuits, Bread, Dal, Milk, Yogurt           5
5          T0006  Bread, Butter, Cereal, Chips, Eggs, Juice, Milk           7
Number of transactions =  500


Apriori algorithm needs the data as a list, so we need to convert the , separated data as a list

In [9]:
transactions = df["Items_Purchased"].apply(lambda x: [item.strip() for item in x.split(",")]).tolist()
print(transactions[:5])

[['Biscuits', 'Bread', 'Butter', 'Milk'], ['Butter', 'Pasta', 'Tomato Sauce'], ['Biscuits', 'Sugar', 'Tea'], ['Bread', 'Butter', 'Milk', 'Pasta', 'Sugar', 'Yogurt'], ['Biscuits', 'Bread', 'Dal', 'Milk', 'Yogurt']]


In [10]:
# One-hot encode the transactions as Ml algos cant directly work with lists
te = TransactionEncoder()
te_data = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_data,columns=te.columns_)
print(basket_df.head(3))


   Apple  Banana  Biscuits  Bread  Butter  Cereal  Cheese  Chips  Coffee  \
0  False   False      True   True    True   False   False  False   False   
1  False   False     False  False    True   False   False  False   False   
2  False   False      True  False   False   False   False  False   False   

   Cooking Oil    Dal   Eggs  Juice   Milk  Pasta   Rice  Sugar    Tea  \
0        False  False  False  False   True  False  False  False  False   
1        False  False  False  False  False   True  False  False  False   
2        False  False  False  False  False  False  False   True   True   

   Tomato Sauce  Yogurt  
0         False   False  
1          True   False  
2         False   False  


In [13]:
#finding frequent itemset
freq_items = apriori(basket_df, min_support=0.10,use_colnames=True)
freq_items = freq_items.sort_values("support",ascending=False)
print(freq_items)

    support                 itemsets
13    0.618                   (Milk)
3     0.540                  (Bread)
30    0.472            (Bread, Milk)
4     0.464                 (Butter)
25    0.362          (Butter, Bread)
..      ...                      ...
22    0.102           (Milk, Banana)
57    0.102     (Butter, Bread, Dal)
32    0.100            (Bread, Rice)
55    0.100     (Apple, Bread, Milk)
56    0.100  (Biscuits, Bread, Milk)

[64 rows x 2 columns]


In [16]:
#Generate association rules
rules = association_rules(freq_items,metric="confidence", min_threshold=0.4).sort_values("lift", ascending=False)
print(rules.head())

       antecedents     consequents  antecedent support  consequent support  \
51        (Coffee)         (Sugar)               0.134               0.284   
27         (Juice)         (Chips)               0.268               0.192   
26         (Chips)         (Juice)               0.192               0.268   
22  (Tomato Sauce)         (Pasta)               0.312               0.180   
21         (Pasta)  (Tomato Sauce)               0.180               0.312   

    support  confidence      lift  representativity  leverage  conviction  \
51    0.110    0.820896  2.890477               1.0  0.071944    3.997667   
27    0.136    0.507463  2.643035               1.0  0.084544    1.640485   
26    0.136    0.708333  2.643035               1.0  0.084544    2.509714   
22    0.148    0.474359  2.635328               1.0  0.091840    1.560000   
21    0.148    0.822222  2.635328               1.0  0.091840    3.870000   

    zhangs_metric   jaccard  certainty  kulczynski  
51       0.7552

Three important metric:
1. Support (helps to select specific product)
2. Confidence (one with another)
3. Lift (A->B): Confidence (A->B) / Support(B)

 Life values:
 Lift >1  : Positive association
 Lift =1  : No meaningful association
 Lift <1  : Negative association